### RAG System from Scratch with Langchain and Python

In [ ]:
# All needed imports

# Data Science Libraries
import faiss
from huggingface_hub import InferenceClient
import numpy as np
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

# Standard Libraries
from dotenv import load_dotenv
import os

In [ ]:
# Chunking

# Loading the document
with open("../../data/raw/rag_notebook.txt") as f:
    knowledge_text = f.read()

# Initializing the Text Splitter, which tries to split on paragraphs ("\n\n"), then newlines ("\n"), then spaces (" "), to keep semantically related text together as much as possible.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,  # Max size of a chunk
    chunk_overlap=20, # Overlap to maintain context between chunks
    length_function=len
)

# Creating the chunks
chunks = text_splitter.split_text(knowledge_text)
print(f"Total number of chunks created: {len(chunks)}")

for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ---\n{chunk}\n")

Total number of chunks created: 21
--- Chunk 1 ---
- Consider these Moon Facts…

--- Chunk 2 ---
It takes 27 1/3 days for the Moon to go around Earth one time; called the Sidereal Month. From new Moon to the next new Moon takes 29 1/2 days;

--- Chunk 3 ---
takes 29 1/2 days; called the Synodic Month. Why this amount of time? Earth has also moved through space and the Moon has to catch up with that

--- Chunk 4 ---
catch up with that starting position. Think of our word month – from the term Moonth.

--- Chunk 5 ---
As the Moon orbits Earth, roughly the same side of the Moon faces Earth. This means that one (1) lunar rotation equals one (1) lunar revolution. Like

--- Chunk 6 ---
revolution. Like Earth, one half (1/2 or 50%) of the Moon is always illuminated by the Sun

--- Chunk 7 ---
- Phases of the Moon

--- Chunk 8 ---
We see the Moon go through its phases due to the position of the Earth, Moon, and Sun relative to each other, NOT due to Clouds, the Moon moving

--- Chunk 9 ---
the

In [4]:
# Embeddings - turning these text chunks into numbers (vectors)

# 'all-MiniLM-L6-v2' is a good, small embedding model, that runs 100% even on not the most demanding local machines
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Embedding all chunks. This will take a moment as the model "reads" and "understands" each chunk.
chunk_embeddings = embedding_model.encode(chunks)

print(f"Shape of the embeddings: {chunk_embeddings.shape}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6471.24it/s]


Shape of the embeddings: (21, 384)


In [5]:
# Vector Store with FAISS
# Now after having created the embeddings vector we need a database to store them in a way we can search by similarity = FAISS

# dimension of our vectors are 384 (the size of the embedding model output)
dims = chunk_embeddings.shape[1]

# Creating the FAISS index
# IndexFlatL2 is the simplest, most basic index - It calculates the exact distance (L2 distance) between our query and all vectors.
index = faiss.IndexFlatL2(dims)

# Adding our chunk embeddings to the index. We must convert to float32 for FAISS
index.add(np.array(chunk_embeddings).astype('float32'))

print(f"FAISS index created with {index.ntotal} vectors.")

FAISS index created with 21 vectors.


In [6]:
load_dotenv()  # loads from .env file in the project root

hf_token = os.getenv("HF_TOKEN")
headers = {"Authorization": f"Bearer {hf_token}"}

client = InferenceClient(token=hf_token)

In [9]:
# Retrieve, Augment, Generate
def answer_question(query, k = 2):
    """Answers a question based on the retrieved context using FAISS and a language model.

    Args:
        query (str): The question to answer.
        k (int, optional): The number of top similar chunks to retrieve. Defaults to 2.

    Returns:
        str: The generated answer.
    """
    # RETRIEVING

    # Embedding the user's query
    query_embedding = embedding_model.encode([query]).astype('float32')

    distances, indices = index.search(query_embedding, k)

    # Getting the actual text chunks from the original 'chunks' list
    retrieved_chunks = [chunks[i] for i in indices[0]]
    context = "\n\n".join(retrieved_chunks)

    print(f"--- RETRIEVED CONTEXT ---\n{context}\n")

    # AUGMENTING + GENERATING
    result = client.chat_completion(
        messages=[{
            "role": "user",
            "content": f"""Answer the following question using *only* the provided context. If the answer is not in the context, say "I don't have that information."

            Context:
            {context}

            Question:
            {query}

            Answer:
            """
        }],
        model="Qwen/Qwen2.5-7B-Instruct", # if this model des not work, use one of them: 'google/flan-t5-small', meta-llama/Meta-Llama-3-8B-Instruct
        max_tokens=150
    )

    return result.choices[0].message.content.strip()

In [10]:
# Asking the Question
query_1 = "Can the moon be seen in the daytime?" # Answer is yes and is present in the data
print(f"Query: {query_1}")
print(f"Answer: {answer_question(query_1)}\n")

Query: Can the moon be seen in the daytime?
--- RETRIEVED CONTEXT ---
- More about the Moon…

Can we see the Moon in the daytime? Yes!

Are there times when we cannot see the Moon at all? Yes, at New Moon and sometimes right before and right after the New Moon because the thin

Answer: Yes!



In [11]:
# Asking the Question that is not present in the data
query_2 = "What is the distance from Earth to the Sun?"
print(f"Query: {query_2}")
print(f"Answer: {answer_question(query_2)}\n")

Query: What is the distance from Earth to the Sun?
--- RETRIEVED CONTEXT ---
miles. Perigee is when the Moon is closest to the Earth and Apogee is when the Moon is farthest from Earth.

Does the Moon keep a constant distance from Earth? No, it varies from a Perigee of about 225,000 miles to an Apogee of about 243,000 miles. Perigee

Answer: I don't have that information.



In [12]:
# Testing the models, cause we need a free tier model
client = InferenceClient(token=os.getenv("HF_TOKEN"))

models_to_test = [
    "meta-llama/Meta-Llama-3-8B-Instruct",
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "mistralai/Mixtral-8x7B-Instruct-v0.1",
    "Qwen/Qwen2.5-7B-Instruct",
    "microsoft/Phi-3-mini-4k-instruct",
    "google/gemma-2-9b-it",
    "HuggingFaceH4/zephyr-7b-beta",
]

for m in models_to_test:
    try:
        result = client.chat_completion(
            messages=[{"role": "user", "content": "Say hello"}],
            model=m,
            max_tokens=10
        )
        print(f"{m}: WORKS - {result.choices[0].message.content}")
    except Exception as e:
        print(f"{m}: FAILED - {str(e)[:80]}")

meta-llama/Meta-Llama-3-8B-Instruct: FAILED - (Request ID: Root=1-6a592791-6ce77373099337637590a2e9;656ddfe6-2a74-42d9-b925-88
meta-llama/Meta-Llama-3.1-8B-Instruct: FAILED - (Request ID: Root=1-6a592791-5e1020c961b736913dcac6b9;fb4d2aec-5525-4c97-8d2c-1b
mistralai/Mistral-7B-Instruct-v0.3: FAILED - (Request ID: Root=1-6a592791-031c8b953dc6ba175047e70d;2e573807-06c7-4ecc-9ab7-aa
mistralai/Mixtral-8x7B-Instruct-v0.1: FAILED - (Request ID: Root=1-6a592792-53b68e8c2a55cb465788dce2;b6eae590-b759-41e3-926e-47
Qwen/Qwen2.5-7B-Instruct: WORKS - Hello! How can I assist you today?
microsoft/Phi-3-mini-4k-instruct: FAILED - (Request ID: Root=1-6a592792-4dc1c8ff35b9618842c67b92;f8bcbf43-bd16-4d6d-b99a-18
google/gemma-2-9b-it: FAILED - (Request ID: Root=1-6a592792-43e67ebd18f15598737c0efa;4032b538-a95a-4854-a14f-43
HuggingFaceH4/zephyr-7b-beta: FAILED - (Request ID: Root=1-6a592792-0acee8d366e2bacf191bbaac;2675f3d9-fc13-4c72-b013-d1
